# Notebook 5 . Cari dan Hapus Duplikat

Temukan baris ganda persis atau baris dengan ID/kombinasi kolom yang sama. Pilih cara menyimpan salah satu salinan. Periksa laporan sebelum memakai hasil.

In [ ]:
from pathlib import Path
import pandas as pd

# KONFIGURASI
INPUT_PATH = Path("Input/input.csv")
SEP = ","
KEY_COLUMNS = ["id"]  # Gunakan [] untuk mencari duplikat baris yang seluruh kolomnya sama
KEEP = "first"        # "first" atau "last"
OUTPUT_PATH = Path("Output/hasil_tanpa_duplikat.csv")
def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Periksa lalu deduplikasi

In [ ]:
source = INPUT_PATH if INPUT_PATH.is_absolute() else BASE_DIR / INPUT_PATH
if not source.exists(): raise FileNotFoundError(f"File tidak ditemukan: {source}")
df = pd.read_csv(source, sep=SEP, dtype="string", encoding="utf-8-sig")
df.columns = df.columns.astype(str).str.strip()
subset = KEY_COLUMNS or None
if subset:
    missing = [col for col in subset if col not in df]
    if missing: raise KeyError(f"Kolom kunci tidak ditemukan: {missing}")
    duplicates = df.duplicated(subset=subset, keep=False)
else:
    duplicates = df.duplicated(keep=False)
print(f"Baris yang termasuk kelompok duplikat: {int(duplicates.sum()):,}")
display(df.loc[duplicates].head(30))
if KEEP not in {"first", "last"}: raise ValueError("KEEP harus 'first' atau 'last'.")
result = df.drop_duplicates(subset=subset, keep=KEEP)
print(f"Baris awal: {len(df):,}; hasil: {len(result):,}; dihapus: {len(df)-len(result):,}")


In [ ]:
output = OUTPUT_PATH if OUTPUT_PATH.is_absolute() else BASE_DIR / OUTPUT_PATH
output.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output, index=False, encoding="utf-8-sig")
print(f"Tersimpan: {output.resolve()}")